# NB07c — ROI Extraction from nnU-Net 80/20 Predictions

**Purpose:** Extract kidney ROIs from new nnU-Net predictions (trained on 166 cases).  
**Input:** `stage1_nnunet_80_20/output/` (208 NIfTI prediction masks)  
**Output:** `kidney_rois_nnunet_80_20/` (ROI crops + metadata)  
**Method:** Anisotropic dilation matching NB03 exactly.

After this: NB07d (Stage 2 training on new ROIs)

In [1]:
# ======== Cell 1: Setup ========
import numpy as np
import nibabel as nib
from pathlib import Path
from scipy.ndimage import binary_dilation, generate_binary_structure, label as scipy_label
import json, time

OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
PREPROC_DIR = OUTPUT_DIR / "preprocessed"
PRED_DIR = OUTPUT_DIR / "stage1_nnunet_80_20" / "output"
ROI_OUT_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"
ROI_OUT_DIR.mkdir(exist_ok=True)

SPLIT_PATH = OUTPUT_DIR / "stats" / "fold_splits_80_20.json"

with open(SPLIT_PATH) as f:
    split_data = json.load(f)
train_cases = sorted(split_data['folds']['fold_0']['train'])
val_cases = sorted(split_data['folds']['fold_0']['val'])
all_cases = sorted(set(train_cases + val_cases))

# Exclude label-quality cases
EXCLUDE = {'case_00015', 'case_00037'}
all_cases = [c for c in all_cases if c not in EXCLUDE]
train_cases = [c for c in train_cases if c not in EXCLUDE]
val_cases = [c for c in val_cases if c not in EXCLUDE]

n_preds = len(list(PRED_DIR.glob("*.nii.gz")))
print(f"Predictions found: {n_preds}")
print(f"Split: {len(train_cases)} train, {len(val_cases)} val, {len(all_cases)} total")
print(f"Excluded: {EXCLUDE}")
print(f"Output: {ROI_OUT_DIR}")

Predictions found: 208
Split: 166 train, 42 val, 208 total
Excluded: {'case_00037', 'case_00015'}
Output: D:\fsdf_uca_kits\outputs\kidney_rois_nnunet_80_20


In [2]:
# ======== Cell 2: Extract ROIs (NB03-style anisotropic dilation) ========
# Matching NB03 exactly:
#   struct_z: 6-connected (generate_binary_structure(3,1)), iterations=3
#   struct_xy: XY-plane only, iterations=13
#   This is NOT true z-only dilation — struct_z also expands XY slightly.
#   But NB03 does it this way, so we match for pipeline consistency.

DILATION_MM = 10.0     # NB03 uses 10mm (verified from source)
SPACING = [3.0, 0.78125, 0.78125]
MIN_COMPONENT_VOXELS = 500

dil_z = max(1, int(round(DILATION_MM / SPACING[0])))    # 3
dil_xy = max(1, int(round(DILATION_MM / SPACING[1])))   # 13

# NB03-style structuring elements
struct_z = generate_binary_structure(3, 1)  # 6-connected (same as NB03)
struct_xy = np.zeros((3, 3, 3), dtype=bool)
struct_xy[1, :, :] = True  # XY plane only

print(f"NB03-style anisotropic dilation: {DILATION_MM}mm")
print(f"  Z: {dil_z} iterations (6-connected struct)")
print(f"  XY: {dil_xy} iterations (XY-plane struct)")
print(f"  Min component: {MIN_COMPONENT_VOXELS} voxels")
print(f"Extracting ROIs from {len(all_cases)} cases...\n")

all_metas = []
missing_cases = []
empty_cases = []
gt_fallback_cases = []
t0 = time.time()

for ci, case_name in enumerate(all_cases):
    case_id = case_name.replace('case_', '')
    pred_path = PRED_DIR / f"KiTS_{case_id}.nii.gz"
    
    if not pred_path.exists():
        missing_cases.append(case_name)
        continue
    
    # Load prediction mask
    pred_data = nib.load(str(pred_path)).get_fdata()
    pred_mask = pred_data.transpose(2, 1, 0).astype(np.uint8)  # XYZ → ZYX
    
    # Load original image and label
    img = np.load(PREPROC_DIR / case_name / "image.npy")
    label = np.load(PREPROC_DIR / case_name / "label.npy")
    D, H, W = img.shape
    
    # Shape safety check
    if pred_mask.shape[0] < D or pred_mask.shape[1] < H or pred_mask.shape[2] < W:
        print(f"  WARNING: {case_name} pred shape {pred_mask.shape} < img shape {img.shape}")
    pred_mask = pred_mask[:D, :H, :W]
    
    # NB03-style anisotropic dilation
    dilated = pred_mask.astype(bool).copy()
    dilated = binary_dilation(dilated, structure=struct_z, iterations=dil_z)
    dilated = binary_dilation(dilated, structure=struct_xy, iterations=dil_xy)
    
    # Connected components → ROIs
    labeled_arr, n_components = scipy_label(dilated.astype(np.int32))
    
    roi_idx = 0
    for comp_id in range(1, n_components + 1):
        comp_mask = (labeled_arr == comp_id)
        if comp_mask.sum() < MIN_COMPONENT_VOXELS:
            continue
        
        coords = np.argwhere(comp_mask)
        z_min, y_min, x_min = coords.min(axis=0)
        z_max, y_max, x_max = coords.max(axis=0) + 1
        
        roi_img = img[z_min:z_max, y_min:y_max, x_min:x_max]
        roi_seg = label[z_min:z_max, y_min:y_max, x_min:x_max]
        
        roi_name = f"{case_name}_roi{roi_idx}"
        np.save(ROI_OUT_DIR / f"{roi_name}_img.npy", roi_img)
        np.save(ROI_OUT_DIR / f"{roi_name}_seg.npy", roi_seg)
        
        has_tumor = bool((roi_seg == 2).any())
        n_tumor = int((roi_seg == 2).sum())
        n_kidney = int((roi_seg >= 1).sum())
        total_tumor_in_case = int((label == 2).sum())
        tumor_recall = n_tumor / max(total_tumor_in_case, 1)
        
        all_metas.append({
            'roi_name': roi_name, 'case_name': case_name,
            'bbox': [int(z_min), int(y_min), int(x_min), int(z_max), int(y_max), int(x_max)],
            'shape': list(roi_img.shape),
            'has_tumor': has_tumor, 'n_tumor': n_tumor, 'n_kidney': n_kidney,
            'tumor_recall': float(tumor_recall),
            'dilation_mm': DILATION_MM,
            'used_gt_fallback': False,
        })
        roi_idx += 1
    
    # GT fallback for TRAINING cases only (not val — that would be cheating)
    if roi_idx == 0:
        is_train = case_name in set(train_cases)
        gt_mask = (label >= 1).astype(bool)
        if is_train and gt_mask.any():
            coords = np.argwhere(gt_mask)
            z_min, y_min, x_min = coords.min(axis=0)
            z_max, y_max, x_max = coords.max(axis=0) + 1
            roi_img = img[z_min:z_max, y_min:y_max, x_min:x_max]
            roi_seg = label[z_min:z_max, y_min:y_max, x_min:x_max]
            roi_name = f"{case_name}_roi0"
            np.save(ROI_OUT_DIR / f"{roi_name}_img.npy", roi_img)
            np.save(ROI_OUT_DIR / f"{roi_name}_seg.npy", roi_seg)
            all_metas.append({
                'roi_name': roi_name, 'case_name': case_name,
                'bbox': [int(z_min), int(y_min), int(x_min), int(z_max), int(y_max), int(x_max)],
                'shape': list(roi_img.shape),
                'has_tumor': bool((roi_seg == 2).any()),
                'n_tumor': int((roi_seg == 2).sum()),
                'n_kidney': int((roi_seg >= 1).sum()),
                'tumor_recall': 1.0,
                'dilation_mm': DILATION_MM,
                'used_gt_fallback': True,
            })
            gt_fallback_cases.append(case_name)
            print(f"  GT FALLBACK: {case_name} (train case, nnU-Net produced 0 ROIs)")
        else:
            empty_cases.append(case_name)
            if not is_train:
                print(f"  WARNING: {case_name} (VAL case) — 0 ROIs, NO fallback (honest eval)")
            else:
                print(f"  WARNING: {case_name} — 0 ROIs, no GT kidney")
    
    if (ci + 1) % 30 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(all_cases)}] {case_name}: {roi_idx} ROIs ({time.time()-t0:.0f}s)")

# Save metadata
with open(ROI_OUT_DIR / "roi_meta_all.json", 'w') as f:
    json.dump(all_metas, f, indent=2)

elapsed = (time.time() - t0) / 60
train_m = [m for m in all_metas if m['case_name'] in set(train_cases)]
val_m = [m for m in all_metas if m['case_name'] in set(val_cases)]
tumor_rois = sum(1 for m in all_metas if m['has_tumor'])

print(f"\nDone in {elapsed:.1f} min")
print(f"Total ROIs: {len(all_metas)} (Train: {len(train_m)}, Val: {len(val_m)})")
print(f"Tumor ROIs: {tumor_rois}")
if missing_cases:
    print(f"Missing predictions: {len(missing_cases)} — {missing_cases[:5]}")
if empty_cases:
    print(f"Empty (0 ROIs, no fallback): {len(empty_cases)} — {empty_cases}")
if gt_fallback_cases:
    print(f"GT fallback (train only): {len(gt_fallback_cases)} — {gt_fallback_cases}")

# Tumor recall check
low_recall = [m for m in all_metas if m['tumor_recall'] < 0.95 and m['has_tumor']]
if low_recall:
    print(f"\nWARNING: {len(low_recall)} ROIs with tumor recall < 95%:")
    for m in low_recall[:10]:
        print(f"  {m['roi_name']}: recall={m['tumor_recall']:.4f}")

NB03-style anisotropic dilation: 10.0mm
  Z: 3 iterations (6-connected struct)
  XY: 13 iterations (XY-plane struct)
  Min component: 500 voxels
Extracting ROIs from 208 cases...

  [1/208] case_00000: 2 ROIs (3s)
  [30/208] case_00030: 2 ROIs (74s)
  [60/208] case_00061: 2 ROIs (156s)
  [90/208] case_00091: 2 ROIs (228s)
  [120/208] case_00121: 2 ROIs (291s)
  [150/208] case_00151: 1 ROIs (366s)
  [180/208] case_00181: 2 ROIs (447s)

Done in 8.9 min
Total ROIs: 504 (Train: 405, Val: 99)
Tumor ROIs: 213

  case_00028_roi0: recall=0.9198
  case_00041_roi0: recall=0.3564
  case_00041_roi1: recall=0.6436
  case_00045_roi0: recall=0.7355
  case_00104_roi0: recall=0.2220
  case_00104_roi1: recall=0.7780
  case_00117_roi0: recall=0.3265
  case_00117_roi1: recall=0.6735
  case_00118_roi0: recall=0.3160
  case_00118_roi1: recall=0.6840


In [3]:
# ======== Cell 3: Verify ROI Quality ========
import json, numpy as np
from pathlib import Path

OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
PREPROC_DIR = OUTPUT_DIR / "preprocessed"
ROI_OUT_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"
SPLIT_PATH = OUTPUT_DIR / "stats" / "fold_splits_80_20.json"

with open(SPLIT_PATH) as f:
    _s = json.load(f)
val_cases = sorted(_s['folds']['fold_0']['val'])

with open(ROI_OUT_DIR / "roi_meta_all.json") as f:
    metas = json.load(f)

# Compare with old nnU-Net ROIs
OLD_ROI_DIR = OUTPUT_DIR / "kidney_rois_nnunet"
old_meta_path = OLD_ROI_DIR / "roi_meta_all.json"
if old_meta_path.exists():
    with open(old_meta_path) as f:
        old_metas = json.load(f)
    print(f"OLD nnU-Net ROIs (70/30): {len(old_metas)} ROIs")
print(f"NEW nnU-Net ROIs (80/20): {len(metas)} ROIs")

# Val set tumor coverage
val_case_set = set(val_cases)
val_metas = [m for m in metas if m['case_name'] in val_case_set]
val_tumor_rois = [m for m in val_metas if m['has_tumor']]
val_cases_with_tumor = set(m['case_name'] for m in val_tumor_rois)

print(f"\nVal set ({len(val_cases)} cases):")
print(f"  ROIs: {len(val_metas)}")
print(f"  Cases with tumor ROIs: {len(val_cases_with_tumor)}")
print(f"  GT fallback used: {sum(1 for m in val_metas if m.get('used_gt_fallback', False))}")

# Check for val cases with tumor but missing from ROIs
all_val_with_tumor = set()
for case_name in val_cases:
    label_path = PREPROC_DIR / case_name / "label.npy"
    if label_path.exists():
        label = np.load(label_path)
        if (label == 2).any():
            all_val_with_tumor.add(case_name)

missed_tumor = all_val_with_tumor - val_cases_with_tumor
if missed_tumor:
    print(f"\n⚠️ Val cases with tumor but NO tumor in ROI: {len(missed_tumor)}")
    for c in sorted(missed_tumor):
        print(f"  {c}")
else:
    print(f"\n✓ All val tumor cases have tumor in extracted ROIs")

# Tumor recall stats
recalls = [m['tumor_recall'] for m in metas if m['has_tumor'] and 'tumor_recall' in m]
if recalls:
    print(f"\nTumor recall (how much GT tumor is inside ROI):")
    print(f"  Mean:   {np.mean(recalls):.4f}")
    print(f"  Min:    {np.min(recalls):.4f}")
    print(f"  <95%:   {sum(1 for r in recalls if r < 0.95)} cases")
    print(f"  100%:   {sum(1 for r in recalls if r >= 0.999)} cases")

# ROI size distribution
shapes = [m['shape'] for m in metas]
depths = [s[0] for s in shapes]
heights = [s[1] for s in shapes]
widths = [s[2] for s in shapes]
print(f"\nROI size distribution:")
print(f"  Depth (Z):  min={min(depths)}, median={np.median(depths):.0f}, max={max(depths)}")
print(f"  Height (Y): min={min(heights)}, median={np.median(heights):.0f}, max={max(heights)}")
print(f"  Width (X):  min={min(widths)}, median={np.median(widths):.0f}, max={max(widths)}")

print(f"\n✓ ROI extraction complete. Next: NB07d (Stage 2 training)")

OLD nnU-Net ROIs (70/30): 432 ROIs
NEW nnU-Net ROIs (80/20): 504 ROIs

Val set (42 cases):
  ROIs: 99
  Cases with tumor ROIs: 42
  GT fallback used: 0

✓ All val tumor cases have tumor in extracted ROIs

Tumor recall (how much GT tumor is inside ROI):
  Mean:   0.9748
  Min:    0.2220
  <95%:   12 cases
  100%:   198 cases

ROI size distribution:
  Depth (Z):  min=4, median=42, max=68
  Height (Y): min=33, median=127, max=261
  Width (X):  min=33, median=124, max=312

✓ ROI extraction complete. Next: NB07d (Stage 2 training)
